# 04 — cross-stage synthesis
triplet-phase-lock pipeline: expand → extend → resist

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8,4)
plt.rcParams['axes.grid'] = True

In [ ]:
def sequence_n(k): return 24.0*k - 25.0
def sequence_n_perturbed(k,a=8.0,p=6.0): return sequence_n(k)+a*np.sin(k/p)
def sequence_n_noisy(k,a=40.0,seed=7):
    rng=np.random.default_rng(seed)
    return sequence_n(k)+rng.normal(0,a,len(k))
def build_triplets(x): return np.stack([x[:-2],x[1:-1],x[2:]],axis=1)
def normalize_rows(x):
    n=np.linalg.norm(x,axis=1,keepdims=True)
    return x/np.maximum(n,1e-12)
def cosine_scores(x,r): return normalize_rows(x)@(r/np.linalg.norm(r))
def direction_change(x):
    d=np.diff(x,axis=0); dn=normalize_rows(d)
    return np.linalg.norm(dn[1:]-dn[:-1],axis=1)
def local_delta(x): return np.linalg.norm(np.diff(x,axis=0),axis=1)

In [ ]:
k=np.arange(1,83)
families={'clean':sequence_n(k),
          'weak':sequence_n_perturbed(k,8,6),
          'strong':sequence_n_perturbed(k,20,4),
          'noisy':sequence_n_noisy(k,40)}
trip={n:build_triplets(v) for n,v in families.items()}

## Expand

In [ ]:
plt.figure()
for n,v in families.items(): plt.plot(k,v,label=n)
plt.title('Expand: structure invariant'); plt.legend(); plt.show()

## Extend

In [ ]:
delta={n:local_delta(t) for n,t in trip.items()}
drift={n:direction_change(t) for n,t in trip.items()}

In [ ]:
plt.figure()
for n in trip: plt.plot(delta[n],label=n)
plt.title('Extend: magnitude'); plt.legend(); plt.show()

In [ ]:
plt.figure()
for n in trip: plt.plot(drift[n],label=n)
plt.title('Extend: directional change'); plt.legend(); plt.show()

## Resist

In [ ]:
ref=normalize_rows(trip['clean']).mean(axis=0); ref=ref/np.linalg.norm(ref)
scores={n:cosine_scores(t,ref) for n,t in trip.items()}
thr45=1/np.sqrt(2); thrS=0.9995
maskS={n:scores[n]>=thrS for n in trip}

In [ ]:
plt.figure()
for n in trip: plt.plot(scores[n],label=n)
plt.axhline(thr45,ls='--'); plt.axhline(thrS,ls='--')
plt.title('Resist: scores'); plt.legend(); plt.show()

In [ ]:
plt.figure()
for n in trip: plt.plot(maskS[n].astype(float),label=n)
plt.title('Resist: strict mask'); plt.legend(); plt.show()

## Cross-stage link

In [ ]:
plt.figure()
for n in trip:
    # drift has length len(triplets)-2, so align it with scores[2:]
    plt.scatter(drift[n], scores[n][2:], label=n)
plt.xlabel('directional change')
plt.ylabel('cosine score')
plt.title('Drift vs Resistance')
plt.legend()
plt.show()
